# Day 10 Content

In [1]:
import asyncio
import httpx
import time
import os

BASE = 'https://dummyjson.com'
print('Ready')

Ready


In [3]:
start = time.perf_counter()
titles = []
for pid in [1,2,3,4,5]:
    r = httpx.get(f'{BASE}/products/{pid}', params={'delay': 500}, timeout=10.0)
    titles.append(r.json()['title'])
print("Got:", titles)
print(f'One after another took {time.perf_counter() - start:.1f}s (~5 x 0.5s = ~2.5s)')

Got: ['Essence Mascara Lash Princess', 'Eyeshadow Palette with Mirror', 'Powder Canister', 'Red Lipstick', 'Red Nail Polish']
One after another took 16.7s (~5 x 0.5s = ~2.5s)


In [5]:
async def get_one_title(pid):
    async with httpx.AsyncClient(base_url=BASE, timeout=15.0) as client:
        r = await client.get(f'/products/{pid}')
        return r.json()['title']

title = await get_one_title(1)
print('Product 1 is:', title)

Product 1 is: Essence Mascara Lash Princess


In [7]:
async def get_titles(pids):
    async with httpx.AsyncClient(base_url=BASE, timeout=15.0) as client:
        async def one(pid):
            r = await client.get(f'products/{pid}', params={'delay':500})
            return r.json()['title']

        jobs = [one(pid) for pid in pids]

        return await asyncio.gather(*jobs)

start = time.perf_counter()
titles = await get_titles([1,2,3,4,5])
print('Got:', titles)
print(f'All at once took  {time.perf_counter() - start:.1f}s (~0.5s, NOT ~2.5s!)')

Got: ['Essence Mascara Lash Princess', 'Eyeshadow Palette with Mirror', 'Powder Canister', 'Red Lipstick', 'Red Nail Polish']
All at once took  6.8s (~0.5s, NOT ~2.5s!)
